# A comparison of holography X-ray phase contrast imaging:

# mean intensity vs intensity correlations

X-ray holography with intensity correlations is given by the following forward problem:
$$F(f)=c_{f} ~~\text{with}~~ c_{f}(x,y):=\operatorname{Cov}(I(x),I(y)),~~\text{and}~~I:=|\mathcal{D}e^{f}u|^2 $$
The forward operator is given by:
$$
F(f):=|\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*|^2.
$$
In contrast to the imaging from intensity correlations, the X-ray holography with mean intensity is given by the following forward problem:
$$G(f)=d_{f} ~~\text{with}~~d_{f}(x):=c_{f}(x,x)=\mathbb{E}[I(x)]^2$$
The forward operator is given by:
$$
G(f):=\operatorname{Diag}(\mathcal{D}M_{\exp(f)}\operatorname{Cov}[u]M_{\exp(f)}^*\mathcal{D}^*).
$$

In [ ]:
import os
import sys

#sys.path.append(os.path.join(os.path.dirname(__file__), '../../'))

from regpy.hilbert import L2
from regpy.vecsps import UniformGridFcts
from regpy.vecsps import NumPyVectorSpace
from regpy.solvers import Setting
from regpy.solvers.nonlinear.fista import FISTA
import regpy.stoprules as rules
from regpy.operators import FresnelPropagator, SquaredModulus
from regpy.functionals import QuadraticNonneg, QuadraticBilateralConstraints
import matplotlib.pyplot as plt

from auxiliary_ops import ReIm, fresnel_prop
from phaseless_passive_ip_ops import Contrast2FactorPhasedCovOp,  Tau, MatrixAutoProductOp, ExpectationCoxModGaussian, CovarianceCoxModGaussian
from create_Vcov import _create_Vcov
from graphic_utils import test_image_cells, test_image_circle_cross, show_measurements, show_results, show_comparison_results, show_measurements

import numpy as np
from numpy.linalg import norm
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)-20s :: %(message)s'
)


# Create forward operators and synthetic data

Setting up parameters 

In [ ]:
#contrast = test_image_circle_cross(N,N)
contrast = np.abs(test_image_cells(circle_strength=0.))
contrast = np.pad(contrast,((64,64),(64,64)),'constant')
N=contrast.shape[0]                     # Pixel number in each spatial direction
scal_factor = complex(0.1,1)
N_frame=3000                      # Number of frames ( or realizations)
T=1e9                             # the observation time or the number of photon counts
#fresnel_number=10/(2*np.pi)       # Fresnel number
fresnel_number = 500
pad_amount = None
Fourier_truncation_amount = None
N_b=4                            # denotes the rank of the matrix V

construct decomposition $\operatorname{Cov}[u]=VV^*$ of the source covariance operator 

In [ ]:
create_types=['spatial', 'Fresnelprop', 'fourier_random']
create_type='fourier_random'
sigma=0.5

grid=UniformGridFcts((-1, 1, N), (-1, 1, N), dtype=complex,periodic=True)
Vcov, S=_create_Vcov(N, N_b, grid=grid)
# here S is the singular values of the matrix V
Vcov = Vcov * (1./S)[(None,None,...)]


In [ ]:
from regpy.operators import Operator
class ScalarMultiplication(Operator):
    r""" Mapping
    x |-> fac x
    from a real to a complex VectorSpaces
    """
    def __init__(self,domain, fac):
        self.fac = fac
        super().__init__(domain.real_space(),domain.complex_space(),linear=True)
    def _eval(self, x):
        return self.fac*x
    def _adjoint(self,z):
        return np.real(np.conj(self.fac)*z)





compute forward operators 

In [ ]:
fp = FresnelPropagator(grid,fresnel_number,pad_amount=100,pad_value=0.,Fourier_truncation_amount=0)
ScalMul = ScalarMultiplication(grid,scal_factor)
Mat_op=Contrast2FactorPhasedCovOp(Vcov,fp) * ScalMul 

# forward operator associated with the intensity correlations
Cov_Cox = CovarianceCoxModGaussian(Mat_op.codomain,fp.codomain,T)
op_intcorr = Cov_Cox * Mat_op 

# forward operator associated with the mean intensity
E_Cox = ExpectationCoxModGaussian(Mat_op.codomain,fp.codomain)
op_meanint = E_Cox * Mat_op 

In [ ]:
grid.coords[0].shape

In [ ]:

im = fp(np.exp(ScalMul(contrast))*Vcov[:,:,0])
fp3 = FresnelPropagator(grid,fresnel_number,pad_amount=100,pad_value=1.,Fourier_truncation_amount=100)
im3 = fp3(np.exp(ScalMul(contrast)))

fig, (ax1,ax2) = plt.subplots(1,2)
im1=ax1.imshow(np.abs(im))
ax1.set_title('without zero padding')
fig.colorbar(im1,ax=ax1)
im2=ax2.imshow(np.real(np.exp(ScalMul(contrast))*Vcov[:,:,0]))
ax2.set_title('with zero padding')
fig.colorbar(im2,ax=ax2)

from numpy.linalg import norm
norm(contrast), norm(im), norm(im3)


Create synthetic intensity data 

In [ ]:
ptw_detection= SquaredModulus(fp.codomain)
taumat_0=Mat_op(contrast) 
data_meanint=ptw_detection.codomain.zeros()   # mean intensity 
intensities=np.zeros((N_frame,)+fp.codomain.shape)

for i in range(0, N_frame):
    random=1/np.sqrt(2)*(np.random.randn(N_b)+complex(0,1)*np.random.randn(N_b))
    uinc=np.tensordot(taumat_0, random, axes=([-1], [0]))
    signal=ptw_detection(uinc)
    
    #Cox-process
    #signal=(1/T)*np.random.poisson(lam=T*signal)
    data_meanint += signal
    intensities[i, :, :]=signal
    
data_meanint /= N_frame

# centering of  intensities
intensities -= data_meanint


In [ ]:
meanint= show_measurements(taumat_0,intensities,data_meanint)
print("relative error in mean intensity:",norm(meanint-data_meanint)/norm(meanint))

# Inversions
Both for mean intensity data and for intensity correlations we use Tikhonov regularization with L^2 data fidelity term and an L^2 penalty term 
incorporating a non-negativity constraint. The regularization parameter is decreased gradually, using the minimizer of the previous Tikhonov functional as initial guess for the next one. Minimizers of the Tikhonov functional are computed by the FISTA algorithm.

## Inversion for mean intensity

initialize inversions

In [ ]:
regpar_meanint = 1e-9    # initial regularization parameter
Nfista_meanint = 50      # number of FISTA steps for each value of the regularization parameter

penalty=QuadraticNonneg(ScalMul.domain)

setting_mean_inten = Setting(op=op_meanint, penalty=penalty, data_fid = L2, 
                                        data_fid_shift=data_meanint,regpar=regpar_meanint)

reco_meanint = np.zeros_like(contrast)   # initial guess is zero
mean_int_stats = []                      # reconstructions and errors for each regularization parameter will stored   

iteratively decrease the regularization parameter

In [ ]:
for _ in range(1):
    setting_mean_inten.regpar = setting_mean_inten.regpar
    FISTA_solver = FISTA(setting_mean_inten,init=reco_meanint,logging_level="INFO")
    stoprule = rules.CountIterations(Nfista_meanint,logging_level="WARN")
    #stoprule = rules.CountIterations(Nfista_meanint,logging_level="WARN")
    reco_meanint, reco_data_meanint=FISTA_solver.run(stoprule)

    error_meanint=norm(reco_meanint-contrast)/norm(contrast)
    print(f"error_meanint = {error_meanint}, alpha= {setting_mean_inten.regpar}")
    mean_int_stats.append([reco_meanint,error_meanint,setting_mean_inten.regpar])
show_results(ScalMul(contrast),ScalMul(reco_meanint))

## Inversion for intensity correlations

initialize inversions

In [ ]:
regpar_intcorr = 1e-9     # initial regularization parameter
Nfista_intcorr = 50       # number of FISTA steps for each value of the regularization parameter

penalty=QuadraticNonneg(ScalMul.domain)

#reco_intcorr = ReIm_op.domain.zeros() # use this for ab initio reconstruction based on intensity correlations
reco_intcorr = reco_meanint # use this for a warm start from mean intensity reconstruction

setting_intcorr = Setting(op=op_intcorr, penalty=penalty, data_fid = L2,
                                            regpar=regpar_intcorr)

intcorr_stats = []

iteratively decrease the regularization parameter

In [ ]:
for _ in range(1):
    setting_intcorr.regpar = setting_intcorr.regpar/3.
    FISTA_solver = FISTA(setting_intcorr,data=intensities,without_codomain_vectors=True,
                        init=reco_intcorr)
    stoprule = (rules.CountIterations(Nfista_intcorr))
    reco_intcorr, reco_data_intcorr=FISTA_solver.run(stoprule)

    error_intcorr=np.linalg.norm(reco_intcorr-contrast)/np.linalg.norm(contrast)
    print(f"error_intcorr = {error_intcorr}, alpha= {setting_intcorr.regpar}",)
    intcorr_stats.append([reco_intcorr,error_intcorr,setting_intcorr.regpar])
show_results(ScalMul(contrast),ScalMul(reco_intcorr))


## Comparative plots

In [ ]:
from graphic_utils import show_comparison_results
show_comparison_results(contrast,reco_intcorr,reco_meanint)

